# LinkedIn Hiring Rate

The LinkedIn Hiring Rate (LHR) measures the number of LinkedIn members who add a new employer to their profile in the same month that their new job begins, divided by the total number of LinkedIn members in that country. LinkedIn uses recent profile updates to support consistent month-to-month comparisons and seasonally adjusts the series to remove recurring seasonal patterns.

The rate is shown as an index whose average month in 2016 equals 1.0. For example, a value of 1.05 means that hiring was 5% higher than the average month in 2016. The index describes hiring activity among LinkedIn members; it is not a count of vacancies or a measure of total employment.

In [1]:
from pathlib import Path
import warnings

import altair as alt
import attaviz
import pandas as pd

attaviz.enable()
alt.data_transformers.enable("vegafusion")
warnings.filterwarnings("ignore")


def find_project_root(marker="pyproject.toml"):
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "LinkedIn"
PROCESSED_PATH = DATA_PATH / "processed"
PROCESSED_PATH.mkdir(exist_ok=True)

LHR_FILE = (
    DATA_PATH / "LinkedIn Hiring Rate" / "LinkedIn_LHR by Industry SA_Aug2026.xlsx"
)
LHR_VALUE = "LHR (SA)"
BASELINE = 1.0

WEST_AFRICA = ["Ghana", "Nigeria"]
COMPARATORS = ["India", "Kenya", "South Africa"]
COUNTRIES = WEST_AFRICA + COMPARATORS

SOURCE_NOTE = (
    "Source: LinkedIn Economic Graph, seasonally adjusted hiring rate "
    "(release Aug 2026).\n"
    "The index is normalised so that the 2016 monthly average equals 1.0."
)

In [2]:
def tidy_lhr(excel_file, sheet_name, names, countries=None):
    """Read one LHR sheet and return a tidy frame."""
    return (
        pd.read_excel(excel_file, sheet_name=sheet_name, header=3)
        .drop(columns=["Unnamed: 0"])
        .set_axis(names, axis="columns")
        .loc[lambda d: d["Country"].ne("Country")]
        .dropna(subset=names)
        .assign(
            Month=lambda d: pd.to_datetime(d["Month"]),
            Country=lambda d: d["Country"].str.strip(),
            **{LHR_VALUE: lambda d: pd.to_numeric(d[LHR_VALUE])},
        )
        .loc[lambda d: d["Country"].isin(countries) if countries else slice(None)]
        .sort_values(names[:-1])
        .reset_index(drop=True)
    )


def baseline_rule(data, value=BASELINE):
    """Draw the 2016 baseline once per chart or panel."""
    return (
        alt.Chart(data)
        .transform_aggregate(_rows="count()")
        .mark_rule(color=attaviz.REFERENCE, strokeDash=[4, 4])
        .encode(y=alt.datum(value))
    )

In [3]:
country_lhr = tidy_lhr(
    LHR_FILE,
    sheet_name="2A - LHR SA by Ctry",
    names=["Month", "Country", LHR_VALUE],
    countries=COUNTRIES,
)
industry_lhr = tidy_lhr(
    LHR_FILE,
    sheet_name="2B - LHR SA by Ctry, Ind",
    names=["Month", "Country", "Industry", LHR_VALUE],
    countries=COUNTRIES,
).assign(Industry=lambda d: d["Industry"].str.strip())

country_lhr.to_csv(PROCESSED_PATH / "lhr_country.csv", index=False)
industry_lhr.to_csv(PROCESSED_PATH / "lhr_industry.csv", index=False)

## Hiring rate by country

In [4]:
countries = sorted(country_lhr["Country"].unique())
zoom = alt.selection_interval(bind="scales", encodings=["x"])
line = (
    alt.Chart(country_lhr)
    .mark_line()
    .encode(
        x=alt.X("Month:T", title=None),
        y=alt.Y(f"{LHR_VALUE}:Q", title="Hiring rate index (2016 = 1.0)"),
        color=alt.Color("Country:N", legend=None),
        tooltip=[
            alt.Tooltip("Country:N"),
            alt.Tooltip("Month:T", format="%b %Y"),
            alt.Tooltip(f"{LHR_VALUE}:Q", format=".2f"),
        ],
    )
    .add_params(zoom)
)
chart = (
    (baseline_rule(country_lhr) + line)
    .properties(width=240, height=170)
    .facet(
        facet=alt.Facet("Country:N", title=None, sort=countries),
        columns=2,
        title="LinkedIn Hiring Rate by country",
    )
    .resolve_scale(y="independent")
)
attaviz.add_caption(chart, SOURCE_NOTE)

alt.VConcatChart(...)

## Hiring rate by industry

Each chart includes every industry reported for that country. Select industries in the legend to toggle their emphasis and clear the selection to show all industries.

In [5]:
def plot_industry_hiring_rate(data, country, width=700, height=340):
    """Plot every reported industry for one country."""
    subset = data.loc[data["Country"].eq(country)].copy()
    industries = sorted(subset["Industry"].unique())
    highlight = alt.selection_point(
        fields=["Industry"],
        bind="legend",
        name="industry",
    )
    zoom = alt.selection_interval(bind="scales", encodings=["x"])

    line = (
        alt.Chart(subset)
        .mark_line()
        .encode(
            x=alt.X("Month:T", title=None),
            y=alt.Y(f"{LHR_VALUE}:Q", title="Hiring rate index (2016 = 1.0)"),
            color=alt.Color(
                "Industry:N",
                sort=industries,
                scale=alt.Scale(scheme="tableau20"),
                legend=alt.Legend(columns=2, symbolLimit=0, labelLimit=320),
            ),
            opacity=alt.when(highlight).then(alt.value(1)).otherwise(alt.value(0.08)),
            tooltip=[
                alt.Tooltip("Industry:N"),
                alt.Tooltip("Month:T", format="%b %Y"),
                alt.Tooltip(f"{LHR_VALUE}:Q", format=".2f"),
            ],
        )
        .add_params(highlight, zoom)
    )
    chart = (baseline_rule(subset) + line).properties(
        width=width,
        height=height,
        title=f"LinkedIn Hiring Rate by industry in {country}",
    )
    return attaviz.add_caption(chart, SOURCE_NOTE)

In [6]:
plot_industry_hiring_rate(industry_lhr, "Ghana")

alt.VConcatChart(...)

In [7]:
plot_industry_hiring_rate(industry_lhr, "Nigeria")

alt.VConcatChart(...)

In [8]:
plot_industry_hiring_rate(industry_lhr, "India")

alt.VConcatChart(...)

In [9]:
plot_industry_hiring_rate(industry_lhr, "Kenya")

alt.VConcatChart(...)

In [10]:
plot_industry_hiring_rate(industry_lhr, "South Africa")

alt.VConcatChart(...)